# WiSig Module-2 — 双分数诊断（类一致性 S_id + 域一致性 S_dom）  
## “头脑风暴”版：多种 S_id / S_dom 公式对比 + TX 随机划分复现

现实系统不会先知道“未知设备”还是“已知设备的新域”。正确流程是对每个样本同时输出：  
- **S_id（类一致性/已知度）**：是否像某个已知 TX  
- **S_dom（域一致性）**：在“像已知”的前提下，它是否落在 source 域支持集（多源、多模态）

本 notebook：
- 随机选择 known/unknown TX（可重复）并重复多次 split；
- 训练 ResNet18-1D 分类器；
- 输出多种 **S_id**（cosine / MSP / energy / entropy / margin / embedding-Mahalanobis）；
- 输出多种 **S_dom**（重点：per-device×per-sourceRX mixture；以及 softmin / mixture NLL / class-weighted mixture NLL）；
- 对比 unknown AUC 与 drift AUC（cross-RX / cross-day，两者都视作“域不一致”）。


In [ ]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# from <your_loader_module> import load_compact_pkl_dataset
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)


TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4

KNOWN_TX_NUM = 4
SOURCE_RX_NUM = 3
TX_SPLIT_REPEATS = 3

TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3

D_DIM = 32
Z_PCA_DIM = 16
TOPK_CLASS_FOR_DOM = 3
BETA_CLASS_WEIGHT = 10.0
TAU_SOFTMIN = 5.0

ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_brainstorm/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(SEED=SEED, dataset_name=dataset_name, dataset_path=dataset_path,
           TX_TOTAL_USE=TX_TOTAL_USE, RX_TOTAL_USE=RX_TOTAL_USE, DAY_TOTAL_USE=DAY_TOTAL_USE,
           KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM, TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
           TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
           Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM, MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
           BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
           IN_PLANES=IN_PLANES, DROPOUT=DROPOUT,
           D_DIM=D_DIM, Z_PCA_DIM=Z_PCA_DIM, TOPK_CLASS_FOR_DOM=TOPK_CLASS_FOR_DOM,
           BETA_CLASS_WEIGHT=BETA_CLASS_WEIGHT, TAU_SOFTMIN=TAU_SOFTMIN)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
print("RUN_DIR:", RUN_DIR)

In [ ]:
def get_idx(lst, val): return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None: return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

In [ ]:
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

rng_rx = np.random.RandomState(SEED)
SOURCE_RXS = list(rng_rx.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()
DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)
print("SOURCE_RXS:", SOURCE_RXS)
print("DRIFT_RX:", DRIFT_RX)

In [ ]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [ ]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train: optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def l2norm(x, axis=1, eps=1e-12):
    return x/(np.linalg.norm(x,axis=axis,keepdims=True)+eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        P[k] = ZN[y==k].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    return l2norm(Z,axis=1) @ P.T

def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex/(ex.sum(axis=1, keepdims=True)+1e-12)

def logsumexp_np(x):
    m = x.max(axis=1, keepdims=True)
    return m + np.log(np.exp(x-m).sum(axis=1, keepdims=True)+1e-12)

def roc_auc(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

In [ ]:
def build_source_train(compact_dataset, KNOWN_TX):
    X_list, y_list, D_list = [], [], []
    rx_id_list = []
    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    X = np.concatenate(X_list,0).astype(np.float32)
    y = np.concatenate(y_list,0).astype(np.int64)
    D = np.concatenate(D_list,0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list,0).astype(np.int64)
    return X,y,D,rx_id

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, y_list, D_list = [], [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64) if tx in KNOWN_TX else np.full((n,), -1, dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
    X = np.concatenate(X_list,0).astype(np.float32)
    y = np.concatenate(y_list,0).astype(np.int64)
    D = np.concatenate(D_list,0).astype(np.float32)
    return X,y,D

In [ ]:
def sid_variants(logits, Z, P, mu_z=None, var_z=None):
    cos = cosine_to_proto(Z, P)
    sid_cos = np.max(cos, axis=1).astype(np.float32)
    k_hat = np.argmax(cos, axis=1).astype(np.int64)

    prob = softmax_np(logits)
    sid_msp = np.max(prob, axis=1).astype(np.float32)

    sid_energy = logsumexp_np(logits).squeeze(1).astype(np.float32)  # -energy

    ent = -np.sum(prob*np.log(prob+1e-12), axis=1).astype(np.float32)
    sid_negent = (-ent).astype(np.float32)

    part = np.partition(prob, -2, axis=1)
    sid_margin = (part[:,-1] - part[:,-2]).astype(np.float32)

    sid_maha = None
    if mu_z is not None and var_z is not None:
        dif = Z[:,None,:] - mu_z[None,:,:]
        dist = np.sum((dif*dif)/(var_z[None,:,:] + 1e-6), axis=2)
        sid_maha = (-np.min(dist, axis=1)).astype(np.float32)

    return {
        "cosine_proto": sid_cos, "MSP": sid_msp, "Energy": sid_energy,
        "NegEntropy": sid_negent, "Margin": sid_margin, "EmbMaha": sid_maha,
        "cos": cos, "k_hat": k_hat, "prob": prob
    }

In [ ]:
def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def dom_V1_minDist(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        vals=[]
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            vals.append(float(mahalanobis_batch(d, mu, Sinv)[0]))
        out[i] = np.min(vals)
    return out

def dom_V1_softminDist(D_eval, k_hat, models_kr, fallback_k, source_rx_ids, tau=5.0):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    tau = float(tau)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        dists=[]
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            dists.append(float(mahalanobis_batch(d, mu, Sinv)[0]))
        dists = np.array(dists, dtype=np.float32)
        m = (-dists/tau).max()
        softmin = -tau*(m + np.log(np.exp((-dists/tau)-m).sum() + 1e-12))
        out[i] = float(softmin)
    return out

def dom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps=[]
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum() + 1e-12) - np.log(R)
        out[i] = float(-loglik)
    return out

def dom_V1_weightedMixNLL(D_eval, cos_scores, models_kr, fallback_k, source_rx_ids, beta=10.0):
    N,K = cos_scores.shape
    pk = softmax_np(beta * cos_scores).astype(np.float32)
    R = len(source_rx_ids)
    out = np.zeros((N,), dtype=np.float32)
    for i in range(N):
        d = D_eval[i:i+1]
        log_terms=[]
        for k in range(K):
            logps=[]
            for rxid in source_rx_ids:
                mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
                logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
            logps = np.array(logps, dtype=np.float32)
            m = logps.max()
            loglik_kr = m + np.log(np.exp(logps-m).sum()+1e-12) - np.log(R)
            log_terms.append(np.log(pk[i,k]+1e-12) + loglik_kr)
        log_terms = np.array(log_terms, dtype=np.float32)
        m = log_terms.max()
        loglik = m + np.log(np.exp(log_terms-m).sum()+1e-12)
        out[i] = float(-loglik)
    return out

def fit_V0_device(D_train, y_train, K, reg=1e-3):
    return {k: fit_gaussian(D_train[y_train==k], reg=reg) for k in range(K)}

def dom_V0_deviceDist(D_eval, k_hat, dev_models):
    out = np.zeros((D_eval.shape[0],), dtype=np.float32)
    for k,(mu,Sinv,logdet) in dev_models.items():
        idx = np.where(k_hat==k)[0]
        if idx.size:
            out[idx] = mahalanobis_batch(D_eval[idx], mu, Sinv)
    return out

def fit_V2_rxPooled(D_train, rx_train, source_rx_ids, reg=1e-3):
    return [fit_gaussian(D_train[rx_train==rxid], reg=reg) for rxid in source_rx_ids]

def dom_V2_rxPooledMinDist(D_eval, rx_models):
    dists = [mahalanobis_batch(D_eval, mu, Sinv) for (mu,Sinv,logdet) in rx_models]
    return np.min(np.stack(dists,1),1).astype(np.float32)

def fit_ZPCA_device_rx(Z_train, y_train, rx_train, K, source_rx_ids, pca_dim=16, reg=1e-3, min_n=20):
    pca = PCA(n_components=min(pca_dim, Z_train.shape[1]), random_state=SEED)
    Zp = pca.fit_transform(Z_train).astype(np.float32)
    models, fb = fit_device_rx_models(Zp, y_train, rx_train, K, source_rx_ids, reg=reg, min_n=min_n)
    return pca, models, fb

def dom_ZPCA_minDist(Z_eval, k_hat, pca, models_kr, fallback_k, source_rx_ids):
    Zp = pca.transform(Z_eval).astype(np.float32)
    return dom_V1_minDist(Zp, k_hat, models_kr, fallback_k, source_rx_ids)

In [ ]:
def run_one_split(split_id, KNOWN_TX, UNKNOWN_TX):
    split_dir = os.path.join(RUN_DIR, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump({"KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX, "SOURCE_RXS": SOURCE_RXS, "DRIFT_RX": DRIFT_RX}, f, indent=2)

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX)
    K = len(KNOWN_TX)

    X_drRX, y_drRX, D_drRX = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_drDAY, y_drDAY, D_drDAY = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)
    X_u, y_u, D_u = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id)
    rows=[]
    for fold,(idx_tr,idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rng = np.random.RandomState(SEED + 1000*split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1*len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(ArrayDataset(X_src[idx_val],   y_src[idx_val]),   batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val=-1.0; best_state=None; patience=0
        hist={"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}
        for ep in range(1,EPOCHS+1):
            tr_loss,tr_acc = run_epoch(model, train_loader, optimizer=opt)
            va_loss,va_acc = run_epoch(model, val_loader, optimizer=None)
            hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
            hist["val_loss"].append(va_loss);   hist["val_acc"].append(va_acc)
            if va_acc > best_val + 1e-4:
                best_val=va_acc
                best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
                patience=0
            else:
                patience += 1
                if patience >= PATIENCE: break

        torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
        with open(os.path.join(fold_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(hist, f, indent=2)
        model.load_state_dict(best_state)

        # stable A (held-out source)
        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        acc = float((np.argmax(logits_A,1) == y_src[idx_te]).mean())
        D_A = D_src[idx_te]

        # drift sets
        logits_B, Z_B = infer_logits_embed(model, X_drRX, batch=512)
        logits_E, Z_E = infer_logits_embed(model, X_drDAY, batch=512)
        logits_U, Z_U = infer_logits_embed(model, X_u, batch=512)

        # prototypes + emb maha
        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        P = prototypes(Z_tr, y_src[idx_train], K)
        mu_z, var_z = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)

        sidA = sid_variants(logits_A, Z_A, P, mu_z=mu_z, var_z=var_z)
        sidU = sid_variants(logits_U, Z_U, P, mu_z=mu_z, var_z=var_z)

        auc_unknown={}
        for key in ["cosine_proto","MSP","Energy","NegEntropy","Margin","EmbMaha"]:
            if sidA[key] is None or sidU[key] is None: 
                continue
            ybin = np.concatenate([np.zeros_like(sidA[key],dtype=np.int64), np.ones_like(sidU[key],dtype=np.int64)])
            score = np.concatenate([-sidA[key], -sidU[key]])
            auc_unknown[key] = roc_auc(ybin, score)

        # domain models fitted on train split
        source_rx_ids = sorted({RX_USE.index(r) for r in SOURCE_RXS})
        V0_dev = fit_V0_device(D_src[idx_train], y_src[idx_train], K)
        V1_kr, V1_fb = fit_device_rx_models(D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20)
        V2_rx = fit_V2_rxPooled(D_src[idx_train], rx_src[idx_train], source_rx_ids, reg=1e-3)

        # optional embedding-PCA domain
        pca_z, Zkr, Zfb = fit_ZPCA_device_rx(Z_tr, y_src[idx_train], rx_src[idx_train], K, source_rx_ids, pca_dim=Z_PCA_DIM, reg=1e-3, min_n=20)

        cos_A = sidA["cos"]; kA = sidA["k_hat"]
        cos_B = cosine_to_proto(Z_B, P); kB = np.argmax(cos_B,1).astype(np.int64)
        cos_E = cosine_to_proto(Z_E, P); kE = np.argmax(cos_E,1).astype(np.int64)

        # compute S_dom variants (same formulas for any drift)
        SdomA={}
        SdomB={}
        SdomE={}

        SdomA["V1_minDist"] = dom_V1_minDist(D_A, kA, V1_kr, V1_fb, source_rx_ids)
        SdomB["V1_minDist"] = dom_V1_minDist(D_drRX, kB, V1_kr, V1_fb, source_rx_ids)
        SdomE["V1_minDist"] = dom_V1_minDist(D_drDAY, kE, V1_kr, V1_fb, source_rx_ids)

        SdomA["V1_softminDist"] = dom_V1_softminDist(D_A, kA, V1_kr, V1_fb, source_rx_ids, tau=TAU_SOFTMIN)
        SdomB["V1_softminDist"] = dom_V1_softminDist(D_drRX, kB, V1_kr, V1_fb, source_rx_ids, tau=TAU_SOFTMIN)
        SdomE["V1_softminDist"] = dom_V1_softminDist(D_drDAY, kE, V1_kr, V1_fb, source_rx_ids, tau=TAU_SOFTMIN)

        SdomA["V1_mixNLL"] = dom_V1_mixNLL(D_A, kA, V1_kr, V1_fb, source_rx_ids)
        SdomB["V1_mixNLL"] = dom_V1_mixNLL(D_drRX, kB, V1_kr, V1_fb, source_rx_ids)
        SdomE["V1_mixNLL"] = dom_V1_mixNLL(D_drDAY, kE, V1_kr, V1_fb, source_rx_ids)

        SdomA["V1_weightedMixNLL"] = dom_V1_weightedMixNLL(D_A, cos_A, V1_kr, V1_fb, source_rx_ids, beta=BETA_CLASS_WEIGHT)
        SdomB["V1_weightedMixNLL"] = dom_V1_weightedMixNLL(D_drRX, cos_B, V1_kr, V1_fb, source_rx_ids, beta=BETA_CLASS_WEIGHT)
        SdomE["V1_weightedMixNLL"] = dom_V1_weightedMixNLL(D_drDAY, cos_E, V1_kr, V1_fb, source_rx_ids, beta=BETA_CLASS_WEIGHT)

        SdomA["V0_deviceDist"] = dom_V0_deviceDist(D_A, kA, V0_dev)
        SdomB["V0_deviceDist"] = dom_V0_deviceDist(D_drRX, kB, V0_dev)
        SdomE["V0_deviceDist"] = dom_V0_deviceDist(D_drDAY, kE, V0_dev)

        SdomA["V2_rxPooledMinDist"] = dom_V2_rxPooledMinDist(D_A, V2_rx)
        SdomB["V2_rxPooledMinDist"] = dom_V2_rxPooledMinDist(D_drRX, V2_rx)
        SdomE["V2_rxPooledMinDist"] = dom_V2_rxPooledMinDist(D_drDAY, V2_rx)

        SdomA["VZ_pca_minDist"] = dom_ZPCA_minDist(Z_A, kA, pca_z, Zkr, Zfb, source_rx_ids)
        SdomB["VZ_pca_minDist"] = dom_ZPCA_minDist(Z_B, kB, pca_z, Zkr, Zfb, source_rx_ids)
        SdomE["VZ_pca_minDist"] = dom_ZPCA_minDist(Z_E, kE, pca_z, Zkr, Zfb, source_rx_ids)

        def auc_drift(a, b):
            ybin = np.concatenate([np.zeros_like(a,dtype=np.int64), np.ones_like(b,dtype=np.int64)])
            return roc_auc(ybin, np.concatenate([a,b]))

        auc_drRX = {k: auc_drift(SdomA[k], SdomB[k]) for k in SdomA.keys()}
        auc_drDAY = {k: auc_drift(SdomA[k], SdomE[k]) for k in SdomA.keys()}

        row={"split":split_id, "fold":fold, "acc":acc,
             "auc_unknown":auc_unknown, "auc_drRX":auc_drRX, "auc_drDAY":auc_drDAY}
        with open(os.path.join(fold_dir, "metrics_brainstorm.json"), "w", encoding="utf-8") as f:
            json.dump(row, f, indent=2)
        rows.append(row)

        print(f"[split {split_id} fold {fold}] acc={acc:.3f}  AUC_u(Energy)={auc_unknown.get('Energy', float('nan')):.3f}  drRX(V1_minDist)={auc_drRX['V1_minDist']:.3f}")

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows=[]
for split_id in range(1, TX_SPLIT_REPEATS+1):
    rng_tx = np.random.RandomState(SEED + 777*split_id)
    tx_perm = list(rng_tx.permutation(TX_USE))
    KNOWN_TX = tx_perm[:KNOWN_TX_NUM]
    UNKNOWN_TX = tx_perm[KNOWN_TX_NUM:]
    print("\n=== TX split", split_id, "===")
    print("KNOWN_TX:", KNOWN_TX)
    print("UNKNOWN_TX:", UNKNOWN_TX)
    all_rows.extend(run_one_split(split_id, KNOWN_TX, UNKNOWN_TX))

with open(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)

print("\nSaved all metrics to:", RUN_DIR)

In [ ]:
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

acc_vals = [r["acc"] for r in all_rows]
unknown_keys = sorted({k for r in all_rows for k in r["auc_unknown"].keys()})
dom_keys = sorted({k for r in all_rows for k in r["auc_drRX"].keys()})

summary={"acc":{}, "unknown":{}, "drRX":{}, "drDAY":{}}
summary["acc"]["mean"], summary["acc"]["std"] = mean_std(acc_vals)

for k in unknown_keys:
    vals=[r["auc_unknown"][k] for r in all_rows if k in r["auc_unknown"]]
    summary["unknown"][k]={"mean": mean_std(vals)[0], "std": mean_std(vals)[1]}

for k in dom_keys:
    valsRX=[r["auc_drRX"][k] for r in all_rows]
    valsDY=[r["auc_drDAY"][k] for r in all_rows]
    summary["drRX"][k]={"mean": mean_std(valsRX)[0], "std": mean_std(valsRX)[1]}
    summary["drDAY"][k]={"mean": mean_std(valsDY)[0], "std": mean_std(valsDY)[1]}

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("=== Mean ± Std over splits×folds ===")
m,s = summary["acc"]["mean"], summary["acc"]["std"]
print(f"acc: {m:.3f} ± {s:.3f}")

print("\n-- Unknown AUC (higher is better) --")
for k in unknown_keys:
    m,s = summary["unknown"][k]["mean"], summary["unknown"][k]["std"]
    print(f"{k:12s}: {m:.3f} ± {s:.3f}")

print("\n-- Drift AUC cross-RX (higher is better) --")
for k in dom_keys:
    m,s = summary["drRX"][k]["mean"], summary["drRX"][k]["std"]
    print(f"{k:18s}: {m:.3f} ± {s:.3f}")

print("\n-- Drift AUC cross-day (higher is better) --")
for k in dom_keys:
    m,s = summary["drDAY"][k]["mean"], summary["drDAY"][k]["std"]
    print(f"{k:18s}: {m:.3f} ± {s:.3f}")